# Phase 6 · S3 (D TCN + Houlsby adapters) generation tuning — same trained model, better decoding (no training)

The same fix that lifted S1 (validation ROUGE-1 0.461 → 0.521 with the same weights) applied to S3 (D TCN + Houlsby adapters): it was submitted with S1's original decoding (minimum length 8% of each chunk, no repetition penalty) — Codabench BERTScore 0.8060 / ROUGE-1 0.351 / ROUGE-L 0.209; validation see S3's Cell 14.

This notebook **does not train anything**. It loads the run's trained weights and prepared chunks from its output dataset (`P6_S3_D_TCN_HoulsbyAdapters/checkpoints/best.pt`, `P6_S3_D_TCN_HoulsbyAdapters/data_cache.pt`) and the run's own model and cached-generation code, adds a repetition penalty (Keskar et al., 2019), and compares decoding settings on 60 held-out validation documents:
- **minimum summary length per chunk:** 15%, 22% or 30% of the chunk's tokens (maximum 50%), × **repetition penalty** 1.2 or 1.35 — around S1's best (22% / 1.2);
- plus the run's original setting (8% / 1.0) for comparison.

The best (ROUGE-1 + ROUGE-L) is re-scored on all 150 validation documents and used for the 100 test summaries. Changing only generation settings is not a new experiment — it is an improved submission of the same run.

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one is enough) · internet not needed.
- Add Input → the run's output dataset **`Phase6_S3_Dataset`**, **`ruchikhemka/data-archd` (Architecture D)** and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 45–90 minutes.

Outputs: `/kaggle/working/P6_S3_tuned/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — The source run's settings (needed to rebuild its model and read its data; training settings are unused here)
# and the decoding settings tuned in this notebook.
SOURCE_RUN = "P6_S3_D_TCN_HoulsbyAdapters"                # the trained run whose checkpoint is re-decoded
RUN_NAME = "P6_S3_tuned"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Chunk-level summarization data.
CHUNK_TOKENS = 1200                 # document tokens per chunk (whole sentences)
MAX_TARGET_TOKENS = 640             # partial-summary tokens per chunk used in training (longer targets are cut)
MAX_CHUNKS_PER_DOC = 40             # chunks kept per document (> 99.5% of documents complete)
SEPARATOR = " Summary:"             # text between the chunk and its summary
VAL_DOCS = 150                      # held-out training documents for validation (never trained on)

# Houlsby adapters: two per TCN block (after the first convolution and after the residual output).
ADAPTER_DIM = 64

# Training.
EPOCHS = 2
BATCH_SEQS = 4                      # sequences (≤ 2,048 tokens) per forward pass, split over both GPUs
GRAD_ACCUM = 4                      # effective batch = 16 sequences
LR = 3e-4                           # adapters start at the identity (W_up = 0)
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.03
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000

# Generation (greedy, cached).
GEN_BATCH = 16
MIN_NEW_FRACTION = 0.08             # minimum summary tokens per chunk = 8% of the chunk's tokens …
MAX_NEW_FRACTION = 0.45             # … maximum = 45% of the chunk's tokens (+ 32), capped at MAX_TARGET_TOKENS
NO_REPEAT_NGRAM = 4
KEEP_FRACTIONS = (0.5, 0.7, 0.85, 1.0)   # share of each chunk summary's sentences kept (tuned on validation)
USE_AMP = DEVICE.type == "cuda"

# Decoding (the only thing tuned here).
MAX_NEW_FRACTION = 0.50                       # maximum summary tokens per chunk = 50% of the chunk (+ 32)
ORIGINAL_SETTING = (0.08, 1.0)                # what the run was submitted with
MIN_NEW_FRACTIONS = (0.15, 0.22, 0.30)        # minimum summary tokens per chunk
REPETITION_PENALTIES = (1.2, 1.35)            # penalty on tokens already generated for the chunk
TUNE_DOCS = 60
print("Run:", RUN_NAME, "| source:", SOURCE_RUN, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Load the source run's prepared data (chunks, held-out validation documents with references, test chunks) and its
# best trained weights; the data settings must equal this notebook's.
src_data = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/data_cache.pt", recursive=True))
src_ckpt = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/checkpoints/best.pt", recursive=True))
assert src_data and src_ckpt, f"Add the output dataset of {SOURCE_RUN} as an input."
data = torch.load(src_data[0], weights_only=False)
assert data["settings"]["chunk_tokens"] == CHUNK_TOKENS and data["settings"]["max_target_tokens"] == MAX_TARGET_TOKENS
best = torch.load(src_ckpt[0], map_location="cpu", weights_only=False)
print(f"{SOURCE_RUN}: {len(data['val'])} validation documents, {len(data['test'])} test documents | best epoch {best['epoch']} "
      f"(validation loss {best['val_loss']:.4f})")

In [ ]:
# Cell 4 — Tokenizer, special ids and the sentence / word patterns used by the source run (decoding and scoring).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SEP_IDS = tokenizer.encode(SEPARATOR).ids
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]
print("Tokenizer ready")

In [ ]:
# Cell 5 — Rebuild the Phase-1 Architecture D TCN and strictly load its pretrained weights.
# Weights and configuration both come from checkpoints/best.pt (keys "model_state_dict" and "model_config").
d_paths = sorted(glob.glob("/kaggle/input/**/architecture_D/checkpoints/best.pt", recursive=True))
assert d_paths, "Add the dataset ruchikhemka/data-archd as an input."
D_CKPT = torch.load(d_paths[0], map_location="cpu", mmap=True, weights_only=False)
D_CONFIG = D_CKPT["model_config"]
assert D_CONFIG["embedding_dim"] == D_CONFIG["channels"] and D_CONFIG["convs_per_block"] == 2

class WeightNormConvParams(nn.Module):
    # Holds the Phase-1 weight-normalised convolution exactly as saved: weight = g * v / ||v||.
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.weight_g = nn.Parameter(torch.ones(channels, 1, 1))
        self.weight_v = nn.Parameter(0.02 * torch.randn(channels, channels, kernel_size))  # overwritten by Phase-1 weights
        self.bias = nn.Parameter(torch.zeros(channels))

    def weight(self):
        return self.weight_g * self.weight_v / self.weight_v.norm(dim=(1, 2), keepdim=True)

class CausalConv1d(nn.Module):
    # Dilated causal convolution computed as one matrix multiply per kernel tap, on [batch, time, channels].
    # Mathematically identical to Phase 1's left-padded Conv1d, but a tap whose shift is >= the sequence length
    # only ever reads padding zeros and is skipped. Sentences are <= 148 positions while dilations reach 512,
    # so this avoids the very slow cuDNN path for large dilations on short inputs.
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.kernel_size, self.dilation = kernel_size, dilation
        self.conv = WeightNormConvParams(channels, kernel_size)

    def forward(self, x):
        T = x.size(1)
        w = self.conv.weight()
        out = self.conv.bias
        for j in range(self.kernel_size):
            shift = (self.kernel_size - 1 - j) * self.dilation
            if shift >= T:
                continue
            shifted = F.pad(x, (0, 0, shift, 0))[:, :T] if shift > 0 else x
            out = out + shifted @ w[:, :, j].t()
        return out

class TCNResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(channels, kernel_size, dilation)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        out = self.dropout1(F.relu(self.conv1(x)))
        out = self.dropout2(F.relu(self.conv2(out)))
        return F.relu(out + x)

class ArchitectureDBackbone(nn.Module):
    # Phase-1 TCN language model without the vocabulary projection.
    # forward(input_ids) returns the state of every token: [batch, tokens, channels].
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.tcn_blocks = nn.ModuleList([TCNResidualBlock(cfg["channels"], cfg["kernel_size"], d, cfg["dropout"])
                                         for d in cfg["dilations"]])

    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for block in self.tcn_blocks:
            x = block(x)
        return x

backbone = ArchitectureDBackbone(D_CONFIG)
backbone.load_state_dict(D_CKPT["model_state_dict"], strict=True)
del D_CKPT

n_backbone = sum(p.numel() for p in backbone.parameters())
assert n_backbone == int(D_CONFIG["estimated_parameter_count"]), "Parameter count differs from Phase-1 config"
receptive_field = 1 + 2 * (D_CONFIG["kernel_size"] - 1) * sum(D_CONFIG["dilations"])
print(json.dumps({k: v for k, v in D_CONFIG.items() if not isinstance(v, (dict, list)) or k == "dilations"}, indent=2))
print(f"Loaded Phase-1 Architecture D from {Path(d_paths[0]).parent} — {n_backbone:,} parameters, "
      f"receptive field {receptive_field} tokens")

In [ ]:
# Cell 6 — Houlsby adapters: x + W_up·GELU(W_down·x) after the first convolution and after the residual output of every
# TCN block (W_up = 0 at start, so the model initially equals Phase 1). The block accepts a padding mask: inputs of every
# convolution are zeroed at padding positions, which equals unpadded causal convolution for the real tokens.
class BottleneckAdapter(nn.Module):
    def __init__(self, channels, bottleneck):
        super().__init__()
        self.adapter_down = nn.Linear(channels, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, channels)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        return x + self.adapter_up(F.gelu(self.adapter_down(x)))

class HoulsbyTCNBlock(nn.Module):
    def __init__(self, block, channels, bottleneck):
        super().__init__()
        self.block = block
        self.adapter_after_conv1 = BottleneckAdapter(channels, bottleneck)
        self.adapter_after_output = BottleneckAdapter(channels, bottleneck)

    def forward(self, x, mask=None):
        m = 1.0 if mask is None else mask
        h = self.adapter_after_conv1(F.relu(self.block.conv1(x * m)))
        h = F.relu(self.block.conv2(h * m))
        return self.adapter_after_output(F.relu(h + x)) * m

class AdaptedTCN(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.token_embedding = base.token_embedding
        self.tcn_blocks = nn.ModuleList([HoulsbyTCNBlock(b, D_CONFIG["channels"], ADAPTER_DIM) for b in base.tcn_blocks])

    def forward(self, input_ids, mask=None):
        x = self.token_embedding(input_ids)
        if mask is not None:
            x = x * mask
        for block in self.tcn_blocks:
            x = block(x, mask)
        return x

for p in backbone.parameters():
    p.requires_grad_(False)
probe_ids = torch.randint(4, 1000, (2, 24))
with torch.no_grad():
    before = backbone.eval()(probe_ids)
adapted_backbone = AdaptedTCN(backbone)
with torch.no_grad():
    assert torch.allclose(before, adapted_backbone(probe_ids), atol=1e-5), "Adapters must start as the identity"
adapted_backbone = adapted_backbone.to(DEVICE)
n_tuned = sum(p.numel() for p in adapted_backbone.parameters() if p.requires_grad)
print(f"Houlsby adapters: 2 × {len(adapted_backbone.tcn_blocks)} blocks — {n_tuned:,} trainable parameters "
      f"({100 * n_tuned / n_backbone:.2f}% of the backbone)")

In [ ]:
# Cell 7 — Prefix language model: logits = final states × tied Phase-1 embedding; the loss covers the summary tokens only.
class SummaryLM(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids, labels):
        """ids, labels [B, T] (labels = next token, -100 where no loss) → (summed cross-entropy, number of target tokens)."""
        states = self.backbone(ids)
        keep = labels.ne(-100)
        logits = F.linear(states[keep], self.backbone.token_embedding.weight).float()
        return F.cross_entropy(logits, labels[keep], reduction="sum").unsqueeze(0), keep.sum().float().unsqueeze(0)

torch.manual_seed(SEED)
model = SummaryLM(adapted_backbone).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = list(TRAINABLE_NAMES)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
CONTEXT = D_CONFIG["context_length"]
assert CHUNK_TOKENS + len(SEP_IDS) + MAX_TARGET_TOKENS + 2 <= CONTEXT, "Chunk + summary must fit in D's training context"
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)} | "
      f"context {CONTEXT} tokens")

In [ ]:
# Cell 8 — Load the source run's trained weights into the rebuilt model (the parameter names must match exactly).
assert set(best["trainable"]) == set(SAVED_NAMES), "Checkpoint does not match this run's trainable parameters"
model.load_state_dict(best["trainable"], strict=False)
model.eval()
print(f"Loaded {len(SAVED_NAMES)} trained tensors from {src_ckpt[0]}")

In [ ]:
# Cell 9 — (the source run's generation code, plus an optional repetition penalty) Exact incremental generation for the TCN: every convolution keeps the history of its (masked) inputs, so a new
# token needs one step per layer; prompts are left-padded with zeroed padding inputs. Greedy, repeated 4-grams blocked.
def conv_weights(conv):
    w = conv.conv.weight()
    return [w[:, :, j].t() for j in range(conv.kernel_size)], conv.conv.bias

def conv_at(conv, weights, history, t):
    """Causal dilated convolution output at position t from the stored input history [B, T_max, C]."""
    taps, bias = weights
    out = bias
    for j, w in enumerate(taps):
        idx = t - (conv.kernel_size - 1 - j) * conv.dilation
        if idx >= 0:
            out = out + history[:, idx] @ w
    return out

def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, prompts, max_new, min_new, repetition_penalty=1.0):
    """prompts: token lists; max_new / min_new: per-prompt limits → generated token lists (without [EOS])."""
    bb.eval()
    B, L, steps = len(prompts), max(len(p) for p in prompts), max(max_new)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, p in enumerate(prompts):
        ids[b, L - len(p):] = torch.tensor(p, device=DEVICE)
    mask = ids.ne(PAD_ID).unsqueeze(-1).float()
    emb = bb.token_embedding.weight
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        # prefill: run the blocks on the whole prompt and keep every convolution's input history
        hist1, hist2, weights = [], [], []
        x = bb.token_embedding(ids) * mask
        for block in bb.tcn_blocks:
            h_in = x * mask
            h = block.adapter_after_conv1(F.relu(block.block.conv1(h_in))) * mask
            y = F.relu(block.block.conv2(h))
            x = block.adapter_after_output(F.relu(y + x)) * mask
            for store, value in ((hist1, h_in), (hist2, h)):
                buffer = torch.zeros(B, L + steps, value.size(-1), device=DEVICE, dtype=value.dtype)
                buffer[:, :L] = value
                store.append(buffer)
            weights.append((conv_weights(block.block.conv1), conv_weights(block.block.conv2)))
        logits = F.linear(x[:, -1], emb).float()
        generated, done = [[] for _ in range(B)], [False] * B
        for step in range(steps):
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            if repetition_penalty != 1.0:                         # CTRL-style penalty on tokens already generated
                for b, seq in enumerate(generated):
                    if seq:
                        idx = torch.tensor(sorted(set(seq)), device=DEVICE)
                        v = logits[b, idx]
                        logits[b, idx] = torch.where(v > 0, v / repetition_penalty, v * repetition_penalty)
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            token = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if token[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(token[b].item())
            if all(done):
                break
            t = L + step
            x = bb.token_embedding(token)
            for block, h1, h2, (w1, w2) in zip(bb.tcn_blocks, hist1, hist2, weights):
                h1[:, t] = x
                h = block.adapter_after_conv1(F.relu(conv_at(block.block.conv1, w1, h1, t)))
                h2[:, t] = h
                y = F.relu(conv_at(block.block.conv2, w2, h2, t))
                x = block.adapter_after_output(F.relu(y + x))
            logits = F.linear(x, emb).float()
    return generated

def summarise(items, min_fraction, repetition_penalty, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + SEP_IDS for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(min_fraction * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new, repetition_penalty)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 10 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 11 — Compare the decoding settings on TUNE_DOCS validation documents (keep all sentences of every chunk summary);
# the first row is the setting the run was submitted with.
tune_items = data["val"][:TUNE_DOCS]
tune_refs = [d["summary"] for d in tune_items]
grid = {}
for min_fraction, penalty in [ORIGINAL_SETTING] + [(m, p) for m in MIN_NEW_FRACTIONS for p in REPETITION_PENALTIES]:
    start_time = time.time()
    summaries = [join_summary(c) for c in summarise(tune_items, min_fraction, penalty)]
    grid[(min_fraction, penalty)] = score_summaries(tune_refs, summaries)
    print(f"  min length {min_fraction:.2f} · repetition penalty {penalty:.2f}: {describe(grid[(min_fraction, penalty)])} "
          f"| {(time.time() - start_time) / 60:.1f} min", flush=True)
BEST_MIN, BEST_PENALTY = max(grid, key=lambda k: grid[k]["rouge1"] + grid[k]["rougeL"])
print(f"Chosen: minimum length {BEST_MIN:.2f} of each chunk, repetition penalty {BEST_PENALTY:.2f} "
      f"(original {ORIGINAL_SETTING}: {grid[ORIGINAL_SETTING]['rouge1']:.4f} / {grid[ORIGINAL_SETTING]['rougeL']:.4f})")

In [ ]:
# Cell 12 — Re-score the chosen setting on all held-out validation documents (comparable with the run's own validation score).
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"], BEST_MIN, BEST_PENALTY)
val_result = score_summaries(refs, [join_summary(c) for c in val_chunk_summaries])
print(f"All {len(refs)} validation documents: {describe(val_result)} | {(time.time() - start_time) / 60:.1f} min")
print("\nExample (validation) — reference:\n", data["val"][0]["summary"][:700],
      "\n\nGenerated:\n", join_summary(val_chunk_summaries[0])[:700])

In [ ]:
# Cell 13 — Test summaries with the chosen setting, format checks, submission.zip and a run summary.
start_time = time.time()
test_chunk_summaries = summarise(data["test"], BEST_MIN, BEST_PENALTY)
prediction = [{"id": d["id"], "summary": join_summary(c)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert len({p["id"] for p in prediction}) == len(prediction) == len(data["test"])
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
json.dump({"source": SOURCE_RUN, "min_new_fraction": BEST_MIN, "repetition_penalty": BEST_PENALTY, "max_new_fraction": MAX_NEW_FRACTION,
           "grid": {f"{k[0]}_{k[1]}": v for k, v in grid.items()}, "validation_all": val_result,
           "test_words_median": int(np.median(words))}, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} (references ≈ 640) "
      f"| generation {(time.time() - start_time) / 60:.1f} min")